Explore NOAA storm events and monthly U.S. Census trade data

This notebook contains reusable examples for the two raw data sources used by the supply-chain-resilience project:

1. NOAA/NCEI Storm Events bulk CSV files (no API token required).
2. U.S. Census International Trade API, monthly imports by port and HS code (API key required).

The examples use only the Python standard library plus `polars`, which is already a project dependency. They default to small downloads; expand the year/month ranges only after checking the returned schema and row counts.

Official documentation: [NOAA Storm Events bulk download](https://www.ncei.noaa.gov/stormevents/ftp.jsp), [NOAA file-format guide](https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/Storm-Data-Bulk-csv-Format.pdf), and [Census monthly International Trade API](https://www.census.gov/data/developers/data-sets/international-trade.html).

## Setup

Copy `.env.example` to `.env` at the repository root and add your Census key:

```text
CENSUS_API_KEY=your_key_here
```

The helper below reads this simple file without adding `python-dotenv` as a dependency. Environment variables that are already set take precedence.

In [2]:
from __future__ import annotations

import functools
import gzip
import io
import json
import os
import re
import time
from datetime import date
from pathlib import Path
from typing import Iterable, Sequence
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode, urljoin
from urllib.request import Request, urlopen

import polars as pl


def find_project_root(start: Path | None = None) -> Path:
    candidate = (start or Path.cwd()).resolve()
    for path in (candidate, *candidate.parents):
        if (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError("Could not find a parent directory containing pyproject.toml")


def load_simple_env(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("\"'"))


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
load_simple_env(PROJECT_ROOT / ".env")

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw-data directory: {RAW_DIR}")


Project root: /Users/wenyi/Documents/ChatGPT/supply-chain-resilience
Raw-data directory: /Users/wenyi/Documents/ChatGPT/supply-chain-resilience/data/raw


In [5]:
USER_AGENT = "supply-chain-resilience/0.1 (research data fetcher)"


def get_bytes(
    url: str,
    *,
    params: dict[str, object] | None = None,
    headers: dict[str, str] | None = None,
    timeout: int = 300,
    max_retries: int = 3,
) -> bytes:
    if params:
        separator = "&" if "?" in url else "?"
        url = f"{url}{separator}{urlencode(params, doseq=True)}"
    request_headers = {"User-Agent": USER_AGENT, **(headers or {})}
    request = Request(url, headers=request_headers)
    for attempt in range(max_retries):
        try:
            with urlopen(request, timeout=timeout) as response:
                return response.read()
        except HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")[:500]
            raise RuntimeError(f"HTTP {exc.code} for {url}: {detail}") from exc
        except (TimeoutError, OSError, URLError) as exc:
            if attempt < max_retries - 1:
                wait = 2 ** attempt
                print(f"  network error (attempt {attempt + 1}/{max_retries}), retrying in {wait}s — {exc}")
                time.sleep(wait)
            else:
                raise RuntimeError(f"Could not reach {url} after {max_retries} attempts: {exc}") from exc
    raise RuntimeError("Unreachable")


def get_json(url: str, *, params: dict[str, object]) -> object:
    raw = get_bytes(url, params=params)
    # Census API returns an empty body (not []) when the query matches no records.
    if not raw.strip():
        raise RuntimeError(f"Census API returned an empty response — check port code, HS code, and date range.\nURL: {url}")
    try:
        return json.loads(raw)
    except json.JSONDecodeError as exc:
        preview = raw[:500].decode("utf-8", errors="replace")
        raise RuntimeError(f"Non-JSON response from Census API: {preview!r}") from exc


## 1. NOAA/NCEI Storm Events

NCEI republishes each data year whenever records are added or corrected, so the creation-date portion of a filename changes. `latest_noaa_file` reads the official directory index and selects the newest file for the requested data year instead of hard-coding a stale filename.

The project's configured qualifying event types are used as defaults. NOAA records all 48 standardized event types from 1996 onward; earlier years have more limited coverage.

In [3]:
NOAA_STORM_INDEX = "https://www.ncei.noaa.gov/pub/data/swdi/stormevents/csvfiles/"
QUALIFYING_EVENT_TYPES = (
    "Hurricane (Typhoon)",
    "Tropical Storm",
    "Storm Surge/Tide",
    "Coastal Flood",
)


# Cached so multi-year pulls fetch the index only once per kernel session.
@functools.lru_cache(maxsize=1)
def _get_noaa_index() -> str:
    return get_bytes(NOAA_STORM_INDEX).decode("utf-8", errors="replace")


def latest_noaa_file(year: int, table: str = "details") -> str:
    if table not in {"details", "fatalities", "locations"}:
        raise ValueError("table must be 'details', 'fatalities', or 'locations'")

    pattern = re.compile(
        rf'href=["\'](StormEvents_{table}-ftp_v1\.0_d{year}_c(\d{{8}})\.csv\.gz)["\']'
    )
    matches = pattern.findall(_get_noaa_index())
    if not matches:
        raise FileNotFoundError(f"No NOAA {table!r} bulk file found for data year {year}")

    filename, _creation_date = max(matches, key=lambda item: item[1])
    return urljoin(NOAA_STORM_INDEX, filename)


def parse_noaa_damage(value: object) -> float | None:
    if value is None:
        return None
    text = str(value).strip().upper().replace(",", "")
    if not text:
        return None
    multipliers = {"K": 1_000.0, "M": 1_000_000.0, "B": 1_000_000_000.0}
    suffix = text[-1] if text[-1].isalpha() else ""
    number = text[:-1] if suffix else text
    try:
        return float(number) * multipliers.get(suffix, 1.0)
    except ValueError:
        return None


def fetch_noaa_storm_events(
    years: Iterable[int],
    *,
    event_types: Sequence[str] | None = QUALIFYING_EVENT_TYPES,
    states: Sequence[str] | None = None,
) -> pl.DataFrame:
    frames: list[pl.DataFrame] = []
    for year in years:
        url = latest_noaa_file(year, table="details")
        print(f"Downloading NOAA Storm Events {year}: {url.rsplit('/', 1)[-1]}")
        csv_bytes = gzip.decompress(get_bytes(url))
        frame = pl.read_csv(
            io.BytesIO(csv_bytes),
            infer_schema_length=10_000,
            ignore_errors=True,
            null_values=["", "NA"],
        )
        frames.append(frame.rename({column: column.lower() for column in frame.columns}))

    if not frames:
        return pl.DataFrame()

    storms = pl.concat(frames, how="diagonal_relaxed")
    if event_types:
        wanted_types = [value.upper() for value in event_types]
        storms = storms.filter(pl.col("event_type").str.to_uppercase().is_in(wanted_types))
    if states:
        wanted_states = [value.upper() for value in states]
        storms = storms.filter(pl.col("state").str.to_uppercase().is_in(wanted_states))

    derived_columns: list[pl.Expr] = []
    if "begin_yearmonth" in storms.columns:
        derived_columns.append(
            pl.col("begin_yearmonth")
            .cast(pl.Utf8)
            .str.strptime(pl.Date, "%Y%m", strict=False)
            .alias("month")
        )
    for source, target in (
        ("damage_property", "property_damage_usd"),
        ("damage_crops", "crop_damage_usd"),
    ):
        if source in storms.columns:
            derived_columns.append(
                pl.col(source)
                .map_elements(parse_noaa_damage, return_dtype=pl.Float64)
                .alias(target)
            )
    return storms.with_columns(derived_columns) if derived_columns else storms


#### Small usage example

In [ ]:
# Small live example: qualifying 2014 events in Louisiana.
# Remove `states` to keep qualifying events nationwide.
storms_example = fetch_noaa_storm_events([2014], states=["LOUISIANA"])
print(storms_example.shape)
storms_example.select(
    [
        "event_id",
        "month",
        "state",
        "event_type",
        "cz_name",
        "property_damage_usd",
        "crop_damage_usd",
    ]
).head(10)

#### Full Download: 2010-2025 All States Events

In [6]:
# Full download qualifying 2000 and 2026 events in all states.
# Remove `states` to keep qualifying events nationwide.
storms_full = fetch_noaa_storm_events(range(2010,2026))
print(storms_full.shape)
storms_full.select(
    [
        "event_id",
        "month",
        "state",
        "event_type",
        "cz_name",
        "property_damage_usd",
        "crop_damage_usd",
    ]
).head(10)

_combined_out = (
    RAW_DIR / "noaa"
    / f"noaa_storm_events_2010_2026_combined_retrieved_{date.today():%Y%m%d}.parquet"
)
storms_full.write_parquet(_combined_out)
print(f"Saved → {_combined_out.name}")

(8499, 54)
Saved → noaa_storm_events_2010_2026_combined_retrieved_20260815.parquet
